# 05 — Model registry

**Début** : charge `04_evaluation` (checkpoint via `03`). **Fin** : LoRA exporté + state.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "mlops").is_dir():
        REPO_ROOT = p
        break
sys.path.insert(0, str(REPO_ROOT))

import torch

from shared.paths import STEP_05_HISTORY, STEP_05_LORA_DIR, STEP_03_CHECKPOINT
from shared.pokemon_dataset import pick_device
from shared.sd_lora_models import DEFAULT_MODEL_ID, build_sd_lora_stack
from shared.step_artifacts import load_previous_step_output, rel_path, resolve_path, save_step_output

prev = load_previous_step_output("05_model_registry")
checkpoint_path = resolve_path(prev["checkpoint_path"])
model_id = prev.get("model_id", DEFAULT_MODEL_ID)

device = pick_device()
vae, unet, text_encoder, noise_scheduler = build_sd_lora_stack(device, model_id=model_id)
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
unet.load_state_dict(ckpt["unet_state_dict"])

STEP_05_LORA_DIR.mkdir(parents=True, exist_ok=True)
unet.save_pretrained(STEP_05_LORA_DIR)
torch.save({
    "train_losses": ckpt["train_losses"],
    "val_losses": ckpt["val_losses"],
    "epochs": ckpt["epoch"],
    "model_id": model_id,
}, STEP_05_HISTORY)

save_step_output("05_model_registry", {
    "checkpoint_path": prev["checkpoint_path"],
    "lora_dir": rel_path(STEP_05_LORA_DIR),
    "training_history_path": rel_path(STEP_05_HISTORY),
    "model_id": model_id,
    "epochs": ckpt["epoch"],
})
print(f"LoRA → {STEP_05_LORA_DIR}")
